# Olist — High-Impact Findings & Hypothesis (Matplotlib)

Charts that back the project's most decision-relevant findings and the validated H1 hypothesis (late delivery hurts satisfaction). Run from the project root (`Project BA Olist/`). All PNGs saved to `06_AI/Outputs/Generated_Charts/`.

**Revenue definition used:** `order_revenue` = merchandise (goods-only, ~R$ 13.2M, matches EDA/AOV R$ 137). `order_revenue_incl_freight` = goods + freight (~R$ 15.4M). Freight tracked separately (`total_freight`).

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

PROJECT = Path.cwd()
MASTER = PROJECT / '02_Cleaned_data' / 'olist_master.csv'
ITEMS  = PROJECT / '02_Cleaned_data' / 'orders_items_aggregated.csv'
PROD   = PROJECT / '02_Cleaned_data' / 'products_clean.csv'
OUT    = PROJECT / '06_AI' / 'Outputs' / 'Generated_Charts'
OUT.mkdir(parents=True, exist_ok=True)

m = pd.read_csv(MASTER, parse_dates=['order_purchase_timestamp',
                                     'order_delivered_customer_date',
                                     'order_estimated_delivery_date'])
m['order_date'] = m['order_purchase_timestamp'].dt.strftime('%Y-%m')

GREEN, RED, NAVY, GRAY = '#0f6b47', '#b0413e', '#1f3a93', '#9aa3ad'
plt.rcParams.update({'figure.facecolor': 'white', 'axes.facecolor': 'white',
                     'axes.grid': True, 'grid.alpha': .3, 'font.size': 10})

def save(name):
    plt.tight_layout(); plt.savefig(OUT / name, dpi=160, bbox_inches='tight'); plt.show()
    print(f'  saved -> {OUT / name}')

print('Loaded:', m.shape)

## 1. Correlation heatmap — which drivers matter

In [ ]:
cols = ['order_revenue','order_revenue_incl_freight','total_freight','item_count',
        'payment_installments_max','delivery_days','days_early_or_late','is_late','review_score']
cm = m[cols].dropna().corr()
import numpy as np
mask = np.triu(np.ones_like(cm, dtype=bool), k=1)
fig, ax = plt.subplots(figsize=(8.5, 6.5))
im = ax.imshow(cm, cmap='RdYlGn', vmin=-1, vmax=1, aspect='auto')
ax.set_xticks(range(len(cm))); ax.set_yticks(range(len(cm)))
ax.set_xticklabels(cm.columns, rotation=45, ha='right'); ax.set_yticklabels(cm.columns)
for i in range(len(cm)):
    for j in range(len(cm)):
        if i >= j:
            ax.text(j, i, f'{cm.iloc[i,j]:.2f}', ha='center', va='center',
                    fontsize=7.5, color='black')
plt.colorbar(im, ax=ax, fraction=0.03, pad=0.03)
ax.set_title('Correlation of key order metrics'); ax.grid(False)
save('viz_01_correlation_heatmap.png')

## 2. H1 dose–response — review score falls as delivery slows

In [ ]:
bins = [-1, 7, 14, 21, 30, np.inf]
labs = ['<=7d', '8-14d', '15-21d', '22-30d', '>30d']
m['bucket'] = pd.cut(m['delivery_days'], bins=bins, labels=labs)
g = m[m['review_score'].notna()].groupby('bucket', observed=True)['review_score'].agg(['mean','count'])
fig, ax = plt.subplots(figsize=(8, 4.5))
bars = ax.bar(g.index, g['mean'], color=[NAVY, NAVY, NAVY, RED, RED], width=0.6)
ax.axhline(4.0, color=GRAY, ls='--', lw=1)
ax.text(4.4, 4.04, 'healthy > 4.0', color=GRAY, ha='right', fontsize=9)
for p, v in zip(bars, g['mean']):
    ax.text(p.get_x()+p.get_width()/2, v+0.05, f'{v:.2f}', ha='center', fontweight='bold')
ax.axvline(2.5, color='k', ls=':', lw=0.8)
ax.text(2.35, 9.4, 'satisfaction collapses ~3 weeks', rotation=90, fontsize=9, color='#444')
ax.set_ylim(0, 5.3); ax.set_ylabel('Average review score (1-5)')
ax.set_xlabel('Delivery time (days)')
ax.set_title('H1 — delivery speed drives satisfaction (dose-response)');
save('viz_02_score_by_delivery_bucket.png')

## 3. The money penalty — on-time vs late review score

In [ ]:
ot = m[m['review_score'].notna()]
means = [ot.loc[ot['is_late']==0,'review_score'].mean(), ot.loc[ot['is_late']==1,'review_score'].mean()]
ns    = [len(ot[ot['is_late']==0]), len(ot[ot['is_late']==1])]
fig, ax = plt.subplots(figsize=(6, 4.5))
b = ax.bar(['On-time', 'Late'], means, color=[NAVY, RED], width=0.5)
for p, v, n in zip(b, means, ns):
    ax.text(p.get_x()+p.get_width()/2, v+0.06, f'{v:.2f}\n(n={n:,})', ha='center', fontweight='bold')
ax.annotate('', xy=(1, 2.57), xytext=(0, 4.29), arrowprops=dict(arrowstyle='->', color='black', lw=1.5))
ax.text(0.5, 3.6, '1.72-star penalty', ha='center', color='black', fontsize=10)
ax.set_ylim(0, 5.3); ax.set_ylabel('Average review score')
ax.set_title('Late delivery costs more than a star (t-test p<0.001)')
save('viz_03_on_vs_late_score.png')

## 4. Growth is volume, not value — orders vs revenue, AOV flat

In [ ]:
mo = m.groupby('order_date').agg(rev=('order_revenue','sum'), n=('order_id','count')).reset_index()
mo['aov'] = mo['rev']/mo['n']
xx = range(len(mo))
fig, ax = plt.subplots(figsize=(9.5, 4.5))
ax.bar(xx, mo['n'], color=GRAY, alpha=0.55, label='Orders')
ax2 = ax.twinx(); ax2.plot(xx, mo['rev']/1e3, color=NAVY, lw=2.4, marker='o', ms=3, label='Revenue (k)')
ax2.plot(xx, mo['aov'], color=RED, lw=1.8, ls='--', label='AOV')
ax.set_xticks(xx, mo['order_date'], rotation=75, fontsize=7)
ax.set_ylabel('Order volume'); ax2.set_ylabel('Revenue (R$ k) / AOV')
ax.legend(loc='upper left'); ax2.legend(loc='upper center')
ax.set_title('Growth is volume-driven (black Friday Nov-2017 spike), AOV flat ~R$ 137')
save('viz_04_monthly_revenue_volume.png')

## 5. Repeat purchase — the critical failure

In [ ]:
cust = m.groupby('customer_unique_id').size()
new = (cust==1).sum(); rep = (cust>1).sum(); total = cust.shape[0]
fig, ax = plt.subplots(figsize=(6, 4.5))
vals = [new, rep]
b = ax.bar(['One purchase', 'Repeat (2+)'], vals, color=[GRAY, NAVY], width=0.5)
for p, v in zip(b, vals): ax.text(p.get_x()+p.get_width()/2, v+500, f'{v:,}', ha='center', fontweight='bold')
ax.text(0.5, vals[0]*0.3, f'{new/total*100:.2f}%', ha='center', color='white', fontweight='bold', fontsize=14)
ax.text(1.5, vals[1]*0.3, f'{rep/total*100:.2f}%', ha='center', color='white', fontweight='bold', fontsize=14)
ax.set_ylabel('Customers'); ax.set_title(f'Repeat rate {rep/total*100:.2f}% — the failing KPI (n={total:,})')
save('viz_05_repeat_rate.png')

## 6. Geography — delivery speed vs satisfaction by state (compounding NE problem)

In [ ]:
st = m.dropna(subset=['review_score']).groupby('customer_state').agg(
    days=('delivery_days','mean'), score=('review_score','mean'), n=('order_id','count')).rename_axis('state').reset_index()
fig, ax = plt.subplots(figsize=(8.5, 5.5))
sc = ax.scatter(st['days'], st['score'], s=st['n']/np.max(st['n'])*600, alpha=0.6, color=NAVY, edgecolors='k', linewidths=0.5)
for _, r in st.sort_values('days', ascending=False).head(6).iterrows():
    ax.annotate(r['state'], (r['days'], r['score']), fontsize=10, fontweight='bold')
for _, r in st.sort_values('score').head(4).iterrows():
    ax.annotate(r['state'], (r['days'], r['score']), fontsize=10, fontweight='bold')
ax.axhline(4.0, color=GRAY, ls='--'); ax.axvline(12.1, color=GRAY, ls='--')
ax.set_xlabel('Avg delivery time (days)'); ax.set_ylabel('Avg review score')
ax.set_title('Worst-delivery states (NE) = worst satisfaction — problems compound (bubble = order volume)')
save('viz_06_state_delivery_vs_score.png')

## 7. Freight burden — 16.6% of revenue carried by customers

In [ ]:
goods = m['order_revenue'].sum(); freight = m['total_freight'].sum()
fig, ax = plt.subplots(figsize=(5.5, 4.5))
labels = ['Merchandise', 'Freight (shipping)']
sizes = [goods, freight]
cols = [NAVY, RED]
w, t, at = ax.pie(sizes, labels=labels, autopct=lambda p: f'{p:.1f}%',
                  colors=cols, startangle=90, explode=(0, 0.05), shadow=False)
for t_, p_ in zip(t, sizes): t_.set_color('white'); t_.set_fontweight('bold')
ax.set_title(f'Freight = {freight/goods*100:.1f}% of merchandise revenue (R$ {freight/1e6:.2f}M)')
save('viz_07_freight_burden.png')

## 8. Revenue concentration — top categories & states

In [ ]:
# Categories (join items x products)
items = pd.read_csv(ITEMS); prod = pd.read_csv(PROD)[['product_id','category_english']]
it = pd.read_csv('02_Cleaned_data/items_clean.csv')[['order_id','product_id']]
cats = m[['order_id','order_revenue']].merge(it, on='order_id').merge(prod, on='product_id')
cat_rev = cats.groupby('category_english')['order_revenue'].sum().sort_values(ascending=False)
cat_top = cat_rev.head(10); cat_rest = cat_rev.iloc[10:].sum()
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4.5))
ax1 = a1.barh(cat_top.index[::-1][:9], cat_top.values[::-1][:9], color=NAVY)
a1.ticklabel_format(axis='x', style='scientific')
a1.set_xlabel('Revenue (BRL)'); a1.set_title('Top categories')
st_rev = m.groupby('customer_state')['order_revenue'].sum().sort_values(ascending=False)
st_top = st_rev.head(5); st_rest = st_rev.iloc[5:].sum()
ax2 = a2.bar(st_top.index, st_top.values, color=GRAY)
ax2 = a2.bar('Other', st_rest, color=RED)
a2.set_ylabel('Revenue (BRL)'); a2.set_title('Top customer states')
fig.suptitle('Concentration: few categories / few states carry the business', y=1.03)
save('viz_08_concentration.png')
